# 01 — Data Understanding

Inspección de los datos reales del FMS (servidor `fms.local`).
Las celdas de este notebook ya fueron ejecutadas una vez contra el servidor
real para confirmar el esquema documentado en el plan — los hallazgos están
documentados en cada sección en vez de quedar como TODO.


In [ ]:
import sys
from pathlib import Path

SRC = Path("..").resolve() / "src"
sys.path.insert(0, str(SRC))

import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)

import json
from fms_client import FmsClient

client = FmsClient()


## 1. Rutas ópticas — forma de la respuesta y paginación

**Hallazgo confirmado:** este FMS tiene solo **4 rutas ópticas** en total
(IDs 41, 60, 104, 117 — `ICA-P3-Independencia`, `ICA-P2-Poroma`,
`RioBlanco-P1-Ayacucho`, `RioBlanco-P2-Chumbao`). Con un universo tan pequeño
no se pudo ejercitar realmente la paginación por `pageNumber` (todas caben en
una sola página), pero sí se confirmó un límite del servidor que no estaba
documentado: **`pageSize` debe estar entre 1 y 100** — pedir 200 devuelve
`HTTP 400 (11002_attribute_out_of_bounds)`. `fms_client.FmsClient.iter_all_optical_routes`
ya usa `page_size=100` por defecto para respetar este límite.

**Hallazgo confirmado (agrupación por RTU):** las 4 rutas no son
independientes entre sí — cada una está monitoreada por una de las **2 RTUs**
(Remote Test Units, los equipos EXFO físicos) registradas en este FMS. El
endpoint `GET /api/topology/remotetestunits` expone esa relación vía
`monitoredAssets.OpticalRoute`, que no estaba wrappeado en `fms_client.py`
(se agregó `FmsClient.list_remote_test_units`). El agrupamiento coincide
exactamente con el prefijo de nombre de cada ruta (`ICA-*` / `RioBlanco-*`),
así que no es casualidad de nomenclatura — es la topología real:

| rtu_id | rtu_name | route_id | route_name |
|---|---|---|---|
| 13 | EAOTHICA | 41 | ICA-P3-Independencia |
| 13 | EAOTHICA | 60 | ICA-P2-Poroma |
| 86 | EAOTHRBL | 104 | RioBlanco-P1-Ayacucho |
| 86 | EAOTHRBL | 117 | RioBlanco-P2-Chumbao |
</cell id="81450bbf">

In [ ]:
page1 = client.list_optical_routes(page_size=100, page=1)
from fms_client import _main as _m
print("total:", _m._total(page1))
obj1 = _m._objects(page1)[0]

for k, v in obj1['opticalRoute'].items():
    print(k, ":", v)
    
for o in _m._objects(page1):
    r = o.get("opticalRoute", o)
    print(r.get("id"), "-", r.get("name"), "-", r.get("type"))


In [ ]:
import importlib
import fms_client
importlib.reload(fms_client)
client = fms_client.FmsClient()
rtu_routes = []

obj2 = next(client.list_remote_test_units())
print("como es un RTU:")
for k, v in obj2.items():
    print(k, ":", v)

for rtu in client.list_remote_test_units():
    monitored = rtu.get("monitoredAssets", {}).get("OpticalRoute", [])
    for route_id in monitored:
        rtu_routes.append({"rtu_id": rtu.get("id"), "rtu_name": rtu.get("name"), "route_id": route_id})

df_rtu_routes = pd.DataFrame(rtu_routes)

routes_by_id = {r.get("id"): r.get("name") for r in client.iter_all_optical_routes()}
df_rtu_routes["route_name"] = df_rtu_routes["route_id"].map(routes_by_id)
df_rtu_routes = df_rtu_routes.sort_values(["rtu_name", "route_id"]).reset_index(drop=True)
df_rtu_routes

## 2. Mediciones (`/v1/results`) — esquema completo de `brief` y `metadata`

**Hallazgo confirmado** (sobre una medición real iOLM de la ruta 41):

- `metadata` trae, entre otros: `AssetId`, `AssetName`, `TestTime`, `TestType`,
  `TestCategory`, `FaultStatus`, **`HasFault`** (booleano, coexiste con
  `FaultStatus` sin contradicciones en la muestra revisada), **`BaselineId`**,
  `PortId`, `PromiseId`, `RelatedResults`.
- `brief.LinkResults.Results[]` trae, por wavelength: `Loss` y `Wavelength`
  **como STRING** (ej. `"18.910"`, `"1650"`), no como número — hay que castear
  explícitamente (`fms_extract._to_float`, ya corregido). Además trae
  `Deviation.Loss` — **el FMS ya calcula su propia desviación vs. el baseline**,
  capturada como `fms_deviation_db_<wavelength>`.
- `brief.Measurement.Elements[]` (solo en mediciones iOLM) es la lista de
  elementos de enlace (conectores, splices, splitters) con su propia pérdida y
  desviación — esto es el dato de "eventos/atenuación por sección" que el
  README mencionaba sin detallar. Capturado como `element_count` y
  `max_element_deviation_db` en `fms_extract.flatten_result`.
- Mediciones `TestType=OTDR` (las "relacionadas" de un iOLM) sí tienen
  `LinkResults.Results` con `Loss` numérico normal, pero no tienen
  `Measurement.Elements`.

**Hallazgo de calidad de datos (importante) — causa raíz confirmada por el equipo de campo:**
una fracción significativa de mediciones reales tiene `Loss="NaN"` (string)
para su único wavelength. **No es un dato faltante aleatorio**: la causa es que
la distancia óptica real de esas rutas excede el alcance confiable del OTDR.
En la ruta **60 (`ICA-P2-Poroma`)**, por ejemplo, el baseline se midió a
~135km, pero esa distancia está más allá del alcance fiable del instrumento —
pequeñas perturbaciones hacen que la distancia detectada varíe entre
mediciones sucesivas, y todo lo que el OTDR reporta más allá de ~110km es
ruido, no señal real. Cuando la medición no logra estabilizarse dentro de ese
rango, el FMS reporta `Loss=NaN` para el link completo.

> **Actualización:** el límite confiable se fijó inicialmente en ~120km, pero
> una revisión posterior de los datos por el usuario confirmó que la distancia
> máxima confiable de los equipos actualmente instalados es de **110km**, no
> 120km. `fms_extract.MAX_RELIABLE_DISTANCE_KM` se ajustó a 110.0 en
> consecuencia (y los datos cacheados se recalcularon con
> `fms_extract.backfill_element_features`). En la muestra histórica actual
> esto no cambió ningún elemento (no hay elementos posicionados entre 110km y
> 120km en los datos ya extraídos), pero el filtro queda correctamente
> ajustado para mediciones futuras.

**Fix aplicado:** en vez de descartar esas mediciones, `fms_extract.py`
filtra `brief.Measurement.Elements[]` por `Position <= MAX_RELIABLE_DISTANCE_KM`
(110km por defecto) y suma la pérdida de los elementos dentro de ese rango
confiable (`elements_loss_sum_db`). `features.build_measurement_features`
usa esa suma como respaldo de `total_loss_db` únicamente cuando el Loss
end-to-end del FMS viene NaN, y marca esas filas con `distance_capped=True`
(incluido como feature en `features.MODEL_FEATURE_COLUMNS`, para que el
modelo pueda aprender que esas mediciones son menos precisas que el resto).
Tras este fix, `total_loss_db` quedó disponible en el 100% de las mediciones
de las 4 rutas (antes: ~98% de NaN en la ruta 60), y `baseline_resolved`
pasó de 0% a ~99.6% en esa ruta. `distance_capped` se activa en 97.9% de la
ruta 60, 46.8% de la ruta 41, 20.5% de la ruta 117 y ~0% de la ruta 104 — el
problema de alcance no es exclusivo de la ruta 60, solo más severo ahí.

`fms_deviation_db` (la desviación que el propio FMS computa) sigue
disponible incluso en mediciones con `Loss=NaN` (~98% de cobertura en la
ruta 60), así que también se mantiene como feature principal en
`features.MODEL_FEATURE_COLUMNS`, complementaria al delta corregido.

In [ ]:
SAMPLE_ROUTE_IDS = [41, 60, 104, 117]

sample = client.get_results(SAMPLE_ROUTE_IDS[2], top=1)
print(json.dumps(sample["results"][0], indent=2, ensure_ascii=False))


### 2.1 Esquema de `metadata` — campo por campo, con ejemplos reales

La celda anterior mostró el JSON completo (truncado) de una medición. Antes de
seguir, vale la pena aislar `metadata` en una tabla — es la parte que
`fms_extract.flatten_result` usa para casi todas sus columnas "planas"
(`route_id`, `route_name`, `TestTime`, etc.), y tiene varios campos que no
estaban documentados en el plan original (`AssetType`, `Filename`,
`TestSetupId`, `HasError`, `EditionSourceId`/`EditionTime`,
`NullingId`/`PairingId`, `PreviousFaultId`).

In [ ]:
sample = client.get_results(104, top=1)
m = sample["results"][0]["metadata"]

rows = []
for k, v in m.items():
    if isinstance(v, list):
        example = f"[lista de {len(v)} dict] (ver sección 6, related results)"
    elif isinstance(v, dict):
        example = "{}" if not v else "{...}"
    elif isinstance(v, str) and len(v) > 55:
        example = v[:52] + "..."
    else:
        example = repr(v)
    rows.append({"campo": k, "ejemplo": example, "tipo_python": type(v).__name__})

df_metadata_schema = pd.DataFrame(rows)
df_metadata_schema

**Hallazgo confirmado:** de los 22 campos de `metadata`, `flatten_result`
sólo usa 11 (`AssetId`→`route_id`, `AssetName`→`route_name`, `PortId`,
`TestTime`, `TestType`, `TestCategory`, `FaultStatus`, `HasFault`,
`BaselineId`, `PromiseId`, más `ResultId`→`resultid` a nivel del resultado).
De los 11 restantes, `RelatedResults` se procesa por separado (sección 6), no
en `flatten_result`. Los otros 10 (`AssetType`, `EditionSourceId`,
`EditionTime`, `FaultValues`, `Filename`, `HasError`, `NullingId`,
`PairingId`, `PreviousFaultId`, `TestSetupId`) no se descartaron por error —
se revisaron y no aportan señal para el modelo de riesgo: son identificadores
internos del FMS (`Filename`, `TestSetupId`, `*Id` de correlación entre
OTDR/iOLM emparejados) o quedaron siempre vacíos/`None` en la muestra
revisada (`EditionSourceId`, `EditionTime`, `NullingId`, `PairingId`,
`PreviousFaultId`, `FaultValues={}`). `AssetType` tampoco aporta: es
constante (`"OpticalRoute"`) en este FMS, ya que no hay otro tipo de asset en
los datos.

### 2.2 Transformación aplicada por `flatten_result` — antes vs. después

El hallazgo de la sección 2 sobre `Loss`/`Wavelength` como STRING aplica
también a `Deviation.Loss`. La celda siguiente muestra el mismo valor antes
(tal como llega del FMS) y después (ya casteado por
`fms_extract.flatten_result`) para una medición real:

In [ ]:
import fms_extract as FE

brief = sample["results"][0]["brief"]
lr = brief["LinkResults"]["Results"][0]

print("=== ANTES (brief.LinkResults.Results[0], tal como llega de la API) ===")
print("Loss:", repr(lr["Loss"]), "->", type(lr["Loss"]).__name__)
print("Wavelength:", repr(lr["Wavelength"]), "->", type(lr["Wavelength"]).__name__)
print("Deviation.Loss:", repr(lr["Deviation"]["Loss"]), "->", type(lr["Deviation"]["Loss"]).__name__)

flat = FE.flatten_result(sample["results"][0])
print()
print("=== DESPUES (fms_extract.flatten_result, columnas resultantes) ===")
print("loss_1650:", repr(flat["loss_1650"]), "->", type(flat["loss_1650"]).__name__)
print("fms_deviation_db_1650:", repr(flat["fms_deviation_db_1650"]), "->", type(flat["fms_deviation_db_1650"]).__name__)

### 2.3 `.head()` del DataFrame ya aplanado (varias mediciones reales, ruta 41)

Por último, así se ve `flatten_result` aplicado a varias mediciones reales
seguidas (no solo una) — es el DataFrame que alimenta a `features.py`. Se
omite `brief_raw` (el JSON completo cacheado, columna de texto) para que la
tabla sea legible.

In [ ]:
data_multi = client.get_results(41, top=5)
# obj3 = [r for r in data_multi["results"]]
# pd_obj3 = pd.DataFrame(obj3).drop(columns=["brief"])
rows_multi = [FE.flatten_result(r) for r in data_multi["results"]]
df_flat_example = pd.DataFrame(rows_multi).drop(columns=["brief_raw"])
# pd_obj3.head()
df_flat_example.head()

## 3. Verificación específica de baseline

**Hallazgo confirmado:** `metadata.BaselineId` aparece en (casi) todas las
mediciones, y se resuelve correctamente contra el historial ya extraído de la
propia ruta en **99.8% de los casos** (`baseline_found`). Sin embargo, el
join exitoso no siempre produce un delta numérico utilizable
(`baseline_resolved`): si la medición *baseline* referenciada tiene ella misma
`Loss="NaN"`, el delta queda en `NaN` aunque el `BaselineId` sí se haya
encontrado. Por eso `features.build_delta_features` expone **dos columnas
separadas**:

- `baseline_found`: el `BaselineId` apuntó a un `resultid` real dentro del
  historial extraído (99.8% global).
- `baseline_resolved`: además ese baseline tiene un `total_loss_db` numérico
  usable para calcular `delta_loss_db` (69.5% global — cae a ~0% en la ruta
  60 específicamente por el problema de `Loss=NaN` de esa ruta, no por una
  falla del join).

No se detectó evidencia de *renovación* de baseline (un mismo `BaselineId`
repetido a lo largo de todo el historial de cada ruta en la muestra revisada),
pero el join por `BaselineId` (en vez de "el más reciente por categoría") se
mantiene como diseño correcto de todas formas, por si ocurre en el futuro.

Para no quedarnos con un solo caso, se repite la verificación con una segunda
ruta — **41 (`ICA-P3-Independencia`)** — usando la misma muestra `top=300`,
para contrastar contra la ruta 60.

In [ ]:
import sys
sys.path.insert(0, "../src")
import fms_extract as FE
import features as F

BASELINE_CHECK_ROUTES = [(60, "ICA-P2-Poroma"), (41, "ICA-P3-Independencia")]

for route_id, route_name in BASELINE_CHECK_ROUTES:
    rows = [FE.flatten_result(r) for r in client.get_results(route_id, top=300).get("results", [])]
    df = pd.DataFrame(rows)
    df_feat = F.build_measurement_features(df)
    lookup = F.build_baseline_lookup(df_feat)
    df_delta = F.build_delta_features(df_feat, lookup)

    print(f"--- ruta {route_id} ({route_name}) ---")
    print("baseline_found:", df_delta["baseline_found"].mean())
    print("baseline_resolved:", df_delta["baseline_resolved"].mean())
    print("total_loss_db NaN rate:", df_delta["total_loss_db"].isna().mean())
    print("fms_deviation_db NaN rate:", df_delta["fms_deviation_db"].isna().mean())
    print()

**Hallazgo confirmado (comparación de las dos rutas):** con la misma muestra
`top=300`, `baseline_found` cae de **19.7%** (ruta 60) a **0%** (ruta 41) — una
diferencia mucho más marcada de lo que sugiere el número global del 99.8%.
No es una falla del join: investigando la causa raíz se confirma que

- la ruta 41 tiene un **único** `BaselineId` para prácticamente todas sus
  mediciones (299 de 300), y ese baseline se capturó el **2025-12-22**
  (`TestCategory=Baseline`, confirmado con
  `client.find_result_by_id(41, "<ese BaselineId>", search_top=3000)`). La
  ruta 41 tiene 2658 mediciones en total, así que esa única medición baseline
  queda muy fuera de la ventana de las 300 más recientes (`$orderby=...desc`),
  y el lookup —construido solo con esa muestra— nunca la encuentra.
- la ruta 60, en cambio, tiene **varios** `BaselineId` distintos a lo largo de
  su historial (fue rebaselineada más de una vez), así que algunos de esos
  baselines caen dentro de la muestra de 300 por azar y sí logran resolverse.

**Conclusión para `02_data_preparation`:** `baseline_found`/`baseline_resolved`
calculados sobre una muestra `top=N` chica **no son representativos** del dato
real — el 99.8% / 69.5% global viene de construir el lookup sobre la
extracción **completa** del historial de cada ruta (vía
`iter_all_results_for_route`, sin tope), no sobre una muestra exploratoria
como la de esta celda. Si en el futuro se necesita resolver un baseline fuera
de una ventana ya extraída, `fms_client.find_result_by_id` ya cubre ese caso.

## 4. Distribución de `TestCategory` / `FaultStatus`

**Hallazgo confirmado** (sobre las 16,839 mediciones extraídas de las 4
rutas): `FaultStatus` está muy repartido entre `Detected` (~53%) y `Cleared`
(~47%), con una minoría `None`/`NotApplicable`. Este es un entorno con una
tasa de fallas reportadas inusualmente alta para ser representativa de un
estado "sano" — probablemente datos de prueba/demo con fallas inducidas
deliberadamente, lo cual conviene tener presente al interpretar las métricas
de los notebooks de modelado (la clase "at_risk" no está tan desbalanceada
como se anticipó en el plan original).


In [ ]:
rows = []
for route_id in SAMPLE_ROUTE_IDS:
    data = client.get_results(route_id, top=300)
    for res in data.get("results", []):
        m = res.get("metadata", {})
        rows.append({"route_id": route_id, "TestCategory": m.get("TestCategory"),
                     "FaultStatus": m.get("FaultStatus")})

df_sample = pd.DataFrame(rows)
display(df_sample["TestCategory"].value_counts())
display(df_sample["FaultStatus"].value_counts())


## 5. Visualización inicial: Loss vs TestTime para una ruta

In [ ]:
import matplotlib.pyplot as plt

route_id = 41  # ICA-P3-Independencia: ruta con Loss utilizable la mayoría del tiempo
assetname = client.get_optical_route(route_id)["opticalRoute"]["name"]
data = client.get_results(route_id, top=300)
print("assetname:", assetname)
pts = []
for res in data.get("results", []):
    m = res.get("metadata", {})
    brief = res.get("brief", {})
    lr = (brief.get("LinkResults") or {}).get("Results", [{}])
    loss = lr[0].get("Loss") if lr else None
    # Loss puede venir como "NaN" o con prefijo de desigualdad (ej. ">13.965")
    # cuando el OTDR reporta una lectura fuera de rango — usar FE._to_float,
    # que ya maneja esos casos (ver fms_extract.py), en vez de float() a mano.
    pts.append({"TestTime": m.get("TestTime"), "Loss": FE._to_float(loss)})

df_pts = pd.DataFrame(pts)
df_pts["TestTime"] = pd.to_datetime(df_pts["TestTime"], errors="coerce")
df_pts = df_pts.sort_values("TestTime")

plt.figure(figsize=(10, 4))
plt.plot(df_pts["TestTime"], df_pts["Loss"], marker="o", markersize=3)
plt.title(f"Loss vs TestTime — ruta {route_id} ({assetname})")
plt.xlabel("TestTime")
plt.ylabel("Loss (dB)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


## 6. Resultados relacionados (`result-related`) — referencia

In [ ]:
data = client.get_results(SAMPLE_ROUTE_IDS[0], top=1)
results = data.get("results", [])
if results:
    sample_result_id = results[0]["resultid"]
    related = client.get_related_results(sample_result_id)
    print(json.dumps(related, indent=2, ensure_ascii=False))


## Esquema confirmado (congelado para `02_data_preparation`)

- `fms_client.iter_all_optical_routes`: `page_size` máximo 100 (límite real del
  servidor).
- `fms_client.iter_all_results_for_route`: **obligatorio** usar paginación con
  `$skip` — confirmado que `/v1/results` limita cada respuesta a 1000 filas
  por defecto, y algunas rutas de este FMS tienen muchas más mediciones
  (ruta 41: 2658, ruta 60: 5129, ruta 104: 8169, ruta 117: 883 — 16,839 en
  total). Sin paginar se pierde silenciosamente la mayoría del historial.
- `fms_extract.flatten_result`: castear `Loss`/`Wavelength`/`Deviation.Loss` a
  float explícitamente (vienen como string). Capturar `BaselineId`,
  `HasFault`, `PortId`, y los elementos de `brief.Measurement.Elements` para
  iOLM.
- `features.build_delta_features`: distinguir `baseline_found` (el join
  encontró el `resultid`) de `baseline_resolved` (ese baseline tiene un valor
  de pérdida numérico utilizable) — son cosas distintas en datos reales.
- **Acción recomendada fuera de este proyecto de analítica:** aunque el
pipeline ya compensa el problema de alcance con `elements_loss_sum_db` /
`distance_capped`, sigue siendo valioso que el equipo de campo revise por qué
el baseline de la ruta 60 (`ICA-P2-Poroma`) se capturó a una distancia
(~135km) más allá del alcance confiable del OTDR (~110km) — podría indicar que
conviene recapturar ese baseline con una configuración de pulso/rango distinta,
en vez de operar permanentemente en el límite del instrumento.